In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df=pd.read_csv("loan_data.csv")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      38084 non-null  float64
 1   person_gender                   34838 non-null  object 
 2   person_education                34838 non-null  object 
 3   person_income                   34786 non-null  float64
 4   person_emp_exp                  40857 non-null  float64
 5   person_home_ownership           40326 non-null  object 
 6   loan_amnt                       39520 non-null  float64
 7   loan_intent                     39520 non-null  object 
 8   loan_int_rate                   39520 non-null  float64
 9   loan_percent_income             39520 non-null  float64
 10  cb_person_cred_hist_length      39520 non-null  float64
 11  credit_score                    40807 non-null  float64
 12  previous_loan_defaults_on_file  

In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

class Preprocessor:
    def __init__(self):
        self.scaler   = StandardScaler()
        self.encoders = {}
    
    def fillna_train(self, df):   
        fill_values = {}
        df=df.copy()

        for col in df.columns:
            if df[col].dtype == 'object':
                fill_values[col] = df[col].mode()[0]
                df[col].fillna(fill_values[col], inplace=True)
            else:
                fill_values[col] = df[col].mean()
                df[col].fillna(fill_values[col], inplace=True)

        return df, fill_values

    def fillna_test(self, df, fill_values):
        df=df.copy()

        for col, value in fill_values.items():
            df[col].fillna(value, inplace=True)

        return df
    
    def encode_train(self, df, threshold=0):
        df = df.copy()
        encoders = {}
        onehot_cols = []

        for col in df.columns:
            if df[col].dtype == 'object':
                if df[col].nunique() <= threshold:
                    onehot_cols.append(col)
                    dummies = pd.get_dummies(df[col], prefix=col, dtype=int)
                    df = pd.concat([df.drop(columns=col), dummies], axis=1)
                else:
                    le = LabelEncoder()
                    df[col] = le.fit_transform(df[col])
                    encoders[col] = le

        return df, encoders, onehot_cols

    def encode_test(self, df, encoders, onehot_cols, train_columns):
        df = df.copy()

        # Label encoding
        for col, le in encoders.items():
            df[col] = df[col].astype(str)
            df[col] = df[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

        # One-hot ONLY for known columns
        for col in onehot_cols:
            dummies = pd.get_dummies(df[col], prefix=col, dtype=int)
            df = pd.concat([df.drop(columns=col), dummies], axis=1)

        # Align columns
        df = df.reindex(columns=train_columns, fill_value=0)

        return df
    
    def scale_train(self, df):
        df = df.copy()
        scalers = {}

        for col in df.columns:
            if df[col].dtype != 'object':
                scaler = StandardScaler()
                df[col] = scaler.fit_transform(df[[col]])
                scalers[col] = scaler

        return df, scalers

    def scale_test(self, df, scalers):
        df = df.copy()

        for col in df.columns:
            if col in scalers:
                df[col] = scalers[col].transform(df[[col]])

        return df

In [4]:
x=df.drop('loan_status', axis=1)
y=df['loan_status']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [5]:
preprocessor = Preprocessor()

x_train_filled, fill_values = preprocessor.fillna_train(df=x_train)
x_test_filled = preprocessor.fillna_test(df=x_test, fill_values=fill_values)

x_train_filled.info()
x_test_filled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36000 entries, 25180 to 15795
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      36000 non-null  float64
 1   person_gender                   36000 non-null  object 
 2   person_education                36000 non-null  object 
 3   person_income                   36000 non-null  float64
 4   person_emp_exp                  36000 non-null  float64
 5   person_home_ownership           36000 non-null  object 
 6   loan_amnt                       36000 non-null  float64
 7   loan_intent                     36000 non-null  object 
 8   loan_int_rate                   36000 non-null  float64
 9   loan_percent_income             36000 non-null  float64
 10  cb_person_cred_hist_length      36000 non-null  float64
 11  credit_score                    36000 non-null  float64
 12  previous_loan_defaults_on_file  3

/var/folders/l4/yvznx4n97gs21nxg5vxxc07r0000gn/T/ipykernel_7602/1296702787.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(fill_values[col], inplace=True)
/var/folders/l4/yvznx4n97gs21nxg5vxxc07r0000gn/T/ipykernel_7602/1296702787.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

In [8]:
x_train_encoded, encoders, onehot_cols = preprocessor.encode_train(df=x_train_filled, threshold=0)
x_test_encoded = preprocessor.encode_test(df=x_test_filled, encoders=encoders, onehot_cols=onehot_cols, train_columns=x_train_encoded.columns)

x_train_encoded.info()
x_test_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36000 entries, 25180 to 15795
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      36000 non-null  float64
 1   person_gender                   36000 non-null  int64  
 2   person_education                36000 non-null  int64  
 3   person_income                   36000 non-null  float64
 4   person_emp_exp                  36000 non-null  float64
 5   person_home_ownership           36000 non-null  int64  
 6   loan_amnt                       36000 non-null  float64
 7   loan_intent                     36000 non-null  int64  
 8   loan_int_rate                   36000 non-null  float64
 9   loan_percent_income             36000 non-null  float64
 10  cb_person_cred_hist_length      36000 non-null  float64
 11  credit_score                    36000 non-null  float64
 12  previous_loan_defaults_on_file  3

In [9]:
x_train_scaled, scalers = preprocessor.scale_train(df=x_train_encoded)
x_test_scaled = preprocessor.scale_test(df=x_test_encoded, scalers=scalers)

x_train_scaled.info()
x_test_scaled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36000 entries, 25180 to 15795
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      36000 non-null  float64
 1   person_gender                   36000 non-null  float64
 2   person_education                36000 non-null  float64
 3   person_income                   36000 non-null  float64
 4   person_emp_exp                  36000 non-null  float64
 5   person_home_ownership           36000 non-null  float64
 6   loan_amnt                       36000 non-null  float64
 7   loan_intent                     36000 non-null  float64
 8   loan_int_rate                   36000 non-null  float64
 9   loan_percent_income             36000 non-null  float64
 10  cb_person_cred_hist_length      36000 non-null  float64
 11  credit_score                    36000 non-null  float64
 12  previous_loan_defaults_on_file  3